# DATA CLEANING
## Hotel Reviews Dataset (TripAdvisor)

Tahap ini bertujuan untuk:
1. Membersihkan missing value
2. Menghapus ulasan kosong atau hanya emoji
3. Menghapus ulasan dengan panjang < 3 kata
4. Menghapus duplicate review (berdasarkan hotel, text, date, username)
5. Menghapus review dengan similarity score ≥ 0.99
6. Menyiapkan dataset final untuk tahap preprocessing


In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


## Load Dataset Gabungan
Dataset yang digunakan merupakan hasil penggabungan review dari Surabaya, Bali, dan Sukabumi.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# Memuat dataset hasil penggabungan
df = pd.read_csv('/content/drive/MyDrive/SKRIPSI_NAVY/Dataset/Dataset_Persiapan/Dataset_EDA/hotel_reviews.csv')
print("Dataset berhasil dimuat.")

Mounted at /content/drive
Dataset berhasil dimuat.


In [ ]:
# Menampilkan contoh data
df.head()

,review_text,review_rating,review_date,url,username,hotel_name,city,hotel_rating,region,word_count,language
0,Pertama kali disini super kagum sama interior ...,5,2025-09-13,https://www.tripadvisor.com/ShowUserReviews-g2...,Deby A,Hotel Majapahit Surabaya - MGallery Collection,Surabaya,4.7,Surabaya,18,id
1,Pelayanan oke banget dibantu dengan resepsioni...,5,2025-09-13,https://www.tripadvisor.com/ShowUserReviews-g2...,Mobile26477862893,Hotel Majapahit Surabaya - MGallery Collection,Surabaya,4.7,Surabaya,12,id
2,Kolam renang dan gym nya sangat bagus dan yang...,5,2025-08-27,https://www.tripadvisor.com/ShowUserReviews-g2...,Farid P,Hotel Majapahit Surabaya - MGallery Collection,Surabaya,4.7,Surabaya,13,id
3,Kami Sekeluarga senang. Kamarnya bersih dan ny...,5,2025-08-27,https://www.tripadvisor.com/ShowUserReviews-g2...,Navigate56583167160,Hotel Majapahit Surabaya - MGallery Collection,Surabaya,4.7,Surabaya,18,id
4,Perbotan kamar seperti AC dan dispenser air mi...,1,2025-08-26,https://www.tripadvisor.com/ShowUserReviews-g2...,robin,Hotel Majapahit Surabaya - MGallery Collection,Surabaya,4.7,Surabaya,15,id


In [ ]:
print("Ukuran Dataset (baris, kolom):")
print(df.shape)

print("\nInformasi Dataset:")
df.info()

Ukuran Dataset (baris, kolom):
(7821, 11)

Informasi Dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7821 entries, 0 to 7820
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   review_text    7821 non-null   object 
 1   review_rating  7821 non-null   int64  
 2   review_date    7821 non-null   object 
 3   url            7821 non-null   object 
 4   username       7821 non-null   object 
 5   hotel_name     7821 non-null   object 
 6   city           7821 non-null   object 
 7   hotel_rating   7821 non-null   float64
 8   region         7821 non-null   object 
 9   word_count     7821 non-null   int64  
 10  language       7821 non-null   object 
dtypes: float64(1), int64(2), object(8)
memory usage: 672.2+ KB


## Standarisasi Struktur Data

### Konversi Tipe Data

Kolom `review_date` diubah menjadi format datetime agar konsisten.

In [ ]:
# Perubahan Tipe Data Kolom review_date ke datetime
df['review_date'] = pd.to_datetime(
    df['review_date'],
    format='mixed'
).dt.normalize()

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7821 entries, 0 to 7820
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   review_text    7821 non-null   object        
 1   review_rating  7821 non-null   int64         
 2   review_date    7821 non-null   datetime64[ns]
 3   url            7821 non-null   object        
 4   username       7821 non-null   object        
 5   hotel_name     7821 non-null   object        
 6   city           7821 non-null   object        
 7   hotel_rating   7821 non-null   float64       
 8   region         7821 non-null   object        
 9   word_count     7821 non-null   int64         
 10  language       7821 non-null   object        
dtypes: datetime64[ns](1), float64(1), int64(2), object(7)
memory usage: 672.2+ KB


## Cek Missing Value


In [ ]:
missing = df.isnull().sum()
missing

,0
review_text,0
review_rating,0
review_date,0
url,0
username,0
hotel_name,0
city,0
hotel_rating,0
region,0
word_count,0


## Menghapus Review Kosong (Missing Review Text)


In [ ]:
df = df.dropna(subset=['review_text'])
print("Ukuran setelah hapus missing review_text:", df.shape)

Ukuran setelah hapus missing review_text: (7821, 11)


## Menghapus Review Kosong atau Hanya Emoji


In [ ]:
import re

df['review_text'] = df['review_text'].astype(str).str.strip()
# Fungsi untuk cek apakah ulasan hanya berisi emoji/simbol/non-huruf
def is_only_emoji_or_symbol(text):
    cleaned = re.sub(r'[A-Za-z0-9]', '', text)
    cleaned = cleaned.strip()
    return (
        len(text) > 0 and
        len(re.sub(r'[A-Za-z0-9]', '', text).strip()) == len(text)
    )
# Cek ulasan yang hanya mengandung emoji dan simbol saja
emoji_only_df = df[df['review_text'].apply(is_only_emoji_or_symbol)]
print("Jumlah ulasan hanya emoji/simbol:", emoji_only_df.shape[0])
emoji_only_df[['hotel_name', 'review_date', 'review_text']].head(10)


Jumlah ulasan hanya emoji/simbol: 0


,hotel_name,review_date,review_text


In [ ]:
# Hapus ulasan yang hanya mengandung emoji atau simbol saja
df = df[~df['review_text'].apply(is_only_emoji_or_symbol)]
df = df[df['review_text'] != ""]
print("Ukuran setelah hapus ulasan yang hanya mengandung emoji/simbol:", df.shape)

Ukuran setelah hapus ulasan yang hanya mengandung emoji/simbol: (7821, 11)


## Menghapus Review dengan Panjang < 3 Kata


In [ ]:
# Hitung jumlah kata
df['word_count'] = df['review_text'].apply(lambda x: len(str(x).split()))

# Cek ulasan yang kurang dari 3 kata
ulasan_pendek = df[df['word_count'] < 3]
print("Daftar Ulasan Pendek (kurang dari 3 kata):")
print(ulasan_pendek[['hotel_name', 'review_date', 'review_text', 'word_count']])

# Hapus ulasan kurang dari 3 Kata
df = df[df['word_count'] >= 3]
print("Ukuran setelah hapus review < 3 kata:", df.shape)

Daftar Ulasan Pendek (kurang dari 3 kata):
Empty DataFrame
Columns: [hotel_name, review_date, review_text, word_count]
Index: []
Ukuran setelah hapus review < 3 kata: (7821, 11)


## Menghapus Data Duplicate Berdasarkan:
hotel_name, review_text, review_date, username


In [ ]:
# Hitung jumlah duplikat berdasarkan review_text
duplicate_count = df.duplicated(subset='review_text').sum()
print("Jumlah ulasan duplikat:", duplicate_count)

Jumlah ulasan duplikat: 0


In [ ]:
# Cek data duplikat berdasarkan kolom hotel_name, review_text, review_date, username
duplicate_mask = df.duplicated(subset=[
    'hotel_name',
    'review_text',
    'review_date',
    'username'
])
print("Jumlah duplicate exact:", duplicate_mask.sum())
df = df[~duplicate_mask]
print("Ukuran setelah hapus duplicate:", df.shape)


Jumlah duplicate exact: 0
Ukuran setelah hapus duplicate: (7821, 11)


## Cek Review dengan Similarity ≥ 0.99
Menggunakan TF-IDF dan Cosine Similarity


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np

texts = df['review_text'].tolist()

print("Membuat TF-IDF matrix...")
vectorizer = TfidfVectorizer(stop_words=None)
tfidf_matrix = vectorizer.fit_transform(texts)

print("Menghitung cosine similarity...")
similarity_matrix = cosine_similarity(tfidf_matrix)
similar_pairs = []
threshold = 0.99

for i in range(len(similarity_matrix)):
    for j in range(i+1, len(similarity_matrix)):
        if similarity_matrix[i][j] >= threshold:
            similar_pairs.append({
                "index_1": i, "index_2": j, "similarity_score": similarity_matrix[i][j],
                "hotel_1": df.iloc[i]['hotel_name'], "hotel_2": df.iloc[j]['hotel_name'],
                "text_1": df.iloc[i]['review_text'], "text_2": df.iloc[j]['review_text'],
                "tanggal_1": df.iloc[i]['review_date'], "tanggal_2": df.iloc[j]['review_date'],
                "username_1": df.iloc[i]['username'], "username_2": df.iloc[j]['username']
            })
similar_df = pd.DataFrame(similar_pairs)
print("Jumlah pasangan similarity >= 0.99:", len(similar_df))
similar_df

Membuat TF-IDF matrix...
Menghitung cosine similarity...
Jumlah pasangan similarity >= 0.99: 0


""


In [ ]:
# Filter pasangan dengan similarity > 0.99
if 'similarity_score' in similar_df.columns and not similar_df.empty:
    high_sim_df = similar_df[similar_df['similarity_score'] > 0.99]
else:
    # If similar_df is empty or 'similarity_score' column doesn't exist, create an empty high_sim_df
    high_sim_df = pd.DataFrame(columns=[
        "index_1", "index_2", "similarity_score", "hotel_1", "hotel_2",
        "text_1", "text_2", "tanggal_1", "tanggal_2", "username_1", "username_2"
    ])
print("Jumlah pasangan > 0.99:", len(high_sim_df))

# Ambil index kedua untuk dihapus
indices_to_drop = set(high_sim_df['index_2']) if not high_sim_df.empty else set()
print("Jumlah data yang akan dihapus:", len(indices_to_drop))

print("\nUlasan yang akan dihapus berdasarkan similarity score:")
if indices_to_drop:
    reviews_to_drop_display = df.iloc[list(indices_to_drop)][['hotel_name', 'review_date', 'review_text']].copy()
    reviews_to_drop_display.insert(0, 'Original Index', df.iloc[list(indices_to_drop)].index)
    display(reviews_to_drop_display)
else:
    print("Tidak ada ulasan yang akan dihapus berdasarkan similarity score.")

# Hapus dari dataframe utama
# Ensure df_cleaned is always defined
if indices_to_drop:
    df_cleaned = df.drop(index=indices_to_drop).reset_index(drop=True)
else:
    df_cleaned = df.copy().reset_index(drop=True) # If no rows to drop, df_cleaned is a copy of df

print("\nUkuran data sebelum dihapus:", len(df))
print("Ukuran data setelah dihapus:", len(df_cleaned))

Jumlah pasangan > 0.99: 0
Jumlah data yang akan dihapus: 0

Ulasan yang akan dihapus berdasarkan similarity score:
Tidak ada ulasan yang akan dihapus berdasarkan similarity score.

Ukuran data sebelum dihapus: 7821
Ukuran data setelah dihapus: 7821


## Simpan Dataset Bersih Setelah Cleaning

In [ ]:
# Drop kolom yang tidak diperlukan
df_cleaned = df_cleaned.drop(columns=['word_count'])

# Reset index setelah cleaning
df_cleaned.reset_index(drop=True, inplace=True)

# Tambahkan kolom ID (mulai dari 1)
df_cleaned.insert(0, 'id', df_cleaned.index + 1)

print("Ukuran final dataset:", df_cleaned.shape)
df_cleaned.head()

Ukuran final dataset: (7821, 11)


,id,review_text,review_rating,review_date,url,username,hotel_name,city,hotel_rating,region,language
0,1,Pertama kali disini super kagum sama interior ...,5,2025-09-13,https://www.tripadvisor.com/ShowUserReviews-g2...,Deby A,Hotel Majapahit Surabaya - MGallery Collection,Surabaya,4.7,Surabaya,id
1,2,Pelayanan oke banget dibantu dengan resepsioni...,5,2025-09-13,https://www.tripadvisor.com/ShowUserReviews-g2...,Mobile26477862893,Hotel Majapahit Surabaya - MGallery Collection,Surabaya,4.7,Surabaya,id
2,3,Kolam renang dan gym nya sangat bagus dan yang...,5,2025-08-27,https://www.tripadvisor.com/ShowUserReviews-g2...,Farid P,Hotel Majapahit Surabaya - MGallery Collection,Surabaya,4.7,Surabaya,id
3,4,Kami Sekeluarga senang. Kamarnya bersih dan ny...,5,2025-08-27,https://www.tripadvisor.com/ShowUserReviews-g2...,Navigate56583167160,Hotel Majapahit Surabaya - MGallery Collection,Surabaya,4.7,Surabaya,id
4,5,Perbotan kamar seperti AC dan dispenser air mi...,1,2025-08-26,https://www.tripadvisor.com/ShowUserReviews-g2...,robin,Hotel Majapahit Surabaya - MGallery Collection,Surabaya,4.7,Surabaya,id


In [ ]:
output_path = "/content/drive/MyDrive/SKRIPSI_NAVY/Dataset/Dataset_Persiapan/Dataset_Cleaned/hotel_reviews_cleaned.xlsx"
df_cleaned.to_excel(output_path, index=False)

output_path = "/content/drive/MyDrive/SKRIPSI_NAVY/Dataset/Dataset_Persiapan/Dataset_Cleaned/hotel_reviews_cleaned.csv"
df_cleaned.to_csv(output_path, index=False, encoding='utf-8')
print("Dataset berhasil disimpan!")


Dataset berhasil disimpan!
